In [0]:

print("==========================================")
print("E-COMMERCE CUSTOMER BEHAVIOUR AND SALES INTELLIGENCE")
print("==========================================")

print("Project: Olist Brazilian E-Commerce")
print("Phase 1: Data Extraction")
print("Layer: Bronze")
print("Status: Initializing...")


E-COMMERCE CUSTOMER BEHAVIOUR AND SALES INTELLIGENCE
Project: Olist Brazilian E-Commerce
Phase 1: Data Extraction
Layer: Bronze
Status: Initializing...


In [0]:

print("Spark Version:", spark.version)

print("Spark Session Initialized Successfully!")


Spark Version: 4.2.0
Spark Session Initialized Successfully!


In [0]:

raw_path = "/Volumes/workspace/default/olist_raw_volume/"

files = dbutils.fs.ls(raw_path)

print("UPLOADED DATASETS")
print("---------------------------")

for file in files:
    print(file.name)

print("---------------------------")
print("Total files uploaded:", len(files))


UPLOADED DATASETS
---------------------------
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv
---------------------------
Total files uploaded: 9


In [0]:

# STEP 1: Read the orders dataset

orders_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(raw_path + "olist_orders_dataset.csv")

# STEP 2: Display first 5 records
display(orders_df.limit(5))

# STEP 3: Display column names and data types
orders_df.printSchema()

# STEP 4: Count total records
print("Total orders:", orders_df.count())


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

Total orders: 99441


In [0]:

# Create Bronze schema
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.bronze
""")

print("Bronze schema created successfully!")

Bronze schema created successfully!


In [0]:

from pyspark.sql.functions import current_timestamp, lit, col

# Location of our uploaded CSV files
raw_path = "/Volumes/workspace/default/olist_raw_volume/"

# Mapping of table names to CSV filenames
datasets = {
    "customers_raw": "olist_customers_dataset.csv",
    "geolocation_raw": "olist_geolocation_dataset.csv",
    "order_items_raw": "olist_order_items_dataset.csv",
    "payments_raw": "olist_order_payments_dataset.csv",
    "reviews_raw": "olist_order_reviews_dataset.csv",
    "orders_raw": "olist_orders_dataset.csv",
    "products_raw": "olist_products_dataset.csv",
    "sellers_raw": "olist_sellers_dataset.csv",
    "category_translation_raw": "product_category_name_translation.csv"
}

# Process each dataset
for table_name, file_name in datasets.items():

    print("Processing:", file_name)

    # Read CSV file
    df = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .csv(raw_path + file_name))
    

# DEBUG: Inspect invalid review scores
if table_name == "reviews_raw":

    from pyspark.sql.functions import col, expr

    print("Checking invalid review scores...")

    invalid_scores = df.filter(
        expr("""
            try_cast(review_score AS INT) IS NULL
            AND review_score IS NOT NULL
        """)
    )

    invalid_scores.select(
        "review_id",
        "review_score",
        "review_creation_date",
        "review_answer_timestamp"
    ).show(20, truncate=False)

    print("Invalid records:", invalid_scores.count())

    
    # DEBUG: Check review_score data type
    if table_name == "reviews_raw":
        print("REVIEW DATASET SCHEMA:")
        df.printSchema()
        
        print("REVIEW SCORE DATA TYPE:")
        print(df.schema["review_score"].dataType)

    
    # Fix review_score data type
    if table_name == "reviews_raw":
        from pyspark.sql.functions import col, expr

        # Find invalid review scores
        invalid_scores = df.filter(
            expr("""
                 try_cast(review_score AS INT) IS NULL
                 AND review_score IS NOT NULL
            """)
            )
        
        print("Invalid review score records:")
        invalid_scores.select(
            "review_id",
            "review_score",
            "review_creation_date",
            "review_answer_timestamp"
        ).show(20, truncate=False)

        print("Number of invalid scores:", invalid_scores.count())
        
        df = (df
              .withColumn("review_score", col("review_score").cast("int"))
              .withColumn("review_creation_date", col("review_creation_date").cast("timestamp"))
              .withColumn("review_answer_timestamp", col("review_answer_timestamp").cast("timestamp"))
        )

        print("Corrected Reviews Schema:")
        df.printSchema()

    # Add ingestion information
    df = (df
          .withColumn("ingestion_timestamp", current_timestamp())
          .withColumn("source_file", lit(file_name))
    )

    if table_name == "reviews_raw":
        df.printSchema()
        print("Review score type:", df.schema["review_score"].dataType)

    # Save as Delta table
    (df.write
     .format("delta")
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable("workspace.bronze." + table_name))


    # Verify record count
    print("Table created:", table_name)
    print("Total records:", df.count())
    print("----------------------------------")

print("ALL NINE BRONZE TABLES CREATED SUCCESSFULLY!")

Processing: olist_customers_dataset.csv
Processing: olist_geolocation_dataset.csv
Processing: olist_order_items_dataset.csv
Processing: olist_order_payments_dataset.csv
Processing: olist_order_reviews_dataset.csv
Processing: olist_orders_dataset.csv
Processing: olist_products_dataset.csv
Processing: olist_sellers_dataset.csv
Processing: product_category_name_translation.csv
ALL NINE BRONZE TABLES CREATED SUCCESSFULLY!


In [0]:
# Verify Bronze review table
reviews = spark.table("workspace.bronze.reviews_raw")

print("Total review records:", reviews.count())
reviews.printSchema()

Total review records: 99224
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)



In [0]:

table_name = "workspace.bronze.reviews_raw"

print("Existing Delta table schema:")

spark.table(table_name).printSchema()

print("Existing review_score type:")

print(
    spark.table(table_name)
    .schema["review_score"]
    .dataType
)


Existing Delta table schema:
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)

Existing review_score type:
IntegerType()


In [0]:

# Display all Bronze tables

tables = spark.sql("SHOW TABLES IN workspace.bronze")

display(tables)

# Count the number of tables
print("Total Bronze tables:", tables.count())

database,tableName,isTemporary
bronze,category_translation_raw,false
bronze,customers_raw,false
bronze,geolocation_raw,false
bronze,order_items_raw,false
bronze,orders_raw,false
bronze,payments_raw,false
bronze,products_raw,false
bronze,reviews_raw,false
bronze,sellers_raw,false


Total Bronze tables: 9


In [0]:

# Verify record counts of all Bronze tables

table_names = [
    "customers_raw",
    "geolocation_raw",
    "order_items_raw",
    "payments_raw",
    "reviews_raw",
    "orders_raw",
    "products_raw",
    "sellers_raw",
    "category_translation_raw"
]

for table in table_names:

    df = spark.table("workspace.bronze." + table)

    print(table, ":", df.count(), "records")


customers_raw : 99441 records
geolocation_raw : 1000163 records
order_items_raw : 112650 records
payments_raw : 103886 records
reviews_raw : 104162 records
orders_raw : 99441 records
products_raw : 32951 records
sellers_raw : 3095 records
category_translation_raw : 71 records
